In [17]:
import matplotlib.pyplot as plt
import numpy as np

In [18]:
L = 1.25 #half of the length of the wheelbase which is 2.5 pixels
r = 0.25 #radius of the wheel which is 0.5 pixels diameter

inverse_kinematics_matrix = np.array([[1/r, 0, L/r], [1/r, 0, -L/r]])

In [19]:
def inverse_kinematics(v_forward, omega):

    desired_vel = np.array([[v_forward], [0], [omega]]) #v_y is 0 due to non sliding and differential robot 

    wheel_velocities = np.dot(inverse_kinematics_matrix, desired_vel)

    phi_1 = wheel_velocities[0][0]
    phi_2 = wheel_velocities[1][0]

    return phi_1, phi_2

In [20]:
v_test = 2
omega_test = 0.5

phi1, phi2 = inverse_kinematics(v_test, omega_test)

print(f"To move forward at {v_test} px/s and turn at {omega_test} rad/s:")
print(f"Wheel 1 (Left) must spin at: {phi1:.2f} rad/s")
print(f"Wheel 2 (Right) must spin at: {phi2:.2f} rad/s")

To move forward at 2 px/s and turn at 0.5 rad/s:
Wheel 1 (Left) must spin at: 10.50 rad/s
Wheel 2 (Right) must spin at: 5.50 rad/s


In [21]:
# Need inverse kinematics in global frame

def inverse_kinematics_global(vx, vy, omega, theta):
    I_P_dot = np.array([[vx], [vy], [omega]])

    R_I_to_0 = np.array([[np.cos(theta), np.sin(theta), 0], [-np.sin(theta), np.cos(theta), 0], [0, 0, 1]])

    local_velocity = np.dot(R_I_to_0, I_P_dot)

    wheel_velocities = np.dot(inverse_kinematics_matrix, local_velocity)

    phi_1 = wheel_velocities[0][0]
    phi_2 = wheel_velocities[1][0]
    return phi_1, phi_2



In [22]:
robot_heading = np.pi / 2
map_vx = 0.0
map_vy = 2.0
map_vtheta = 0.0

phi1, phi2 = inverse_kinematics_global(map_vx, map_vy, map_vtheta, robot_heading)

print(f"To move global up while facing up:")
print(f"Wheel 1: {phi1:.2f} rad/s")
print(f"Wheel 2: {phi2:.2f} rad/s")

To move global up while facing up:
Wheel 1: 8.00 rad/s
Wheel 2: 8.00 rad/s


In [23]:
robot_heading = np.pi / 2
map_vx = 2.0
map_vy = 2.0
map_vtheta = 0.0

phi1, phi2 = inverse_kinematics_global(map_vx, map_vy, map_vtheta, robot_heading)

print(f"To move global diagonal while facing up:")
print(f"Wheel 1: {phi1:.2f} rad/s")
print(f"Wheel 2: {phi2:.2f} rad/s")

To move global diagonal while facing up:
Wheel 1: 8.00 rad/s
Wheel 2: 8.00 rad/s


Makes sense as this is verifying the non sliding condition -> cant do this so we need to rotate first then drive

In [24]:
def forward_kinematics(phi_1, phi_2, theta):
    phi_dot = np.array([[phi_1], [phi_2]])

    J_forward = np.array([[r/2, r/2], [0, 0], [r/(2*L), -r/(2*L)]])

    R_0_to_I = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])

    local_velocity = np.dot(J_forward, phi_dot)
    global_velocity = np.dot(R_0_to_I, local_velocity)

    vx = global_velocity[0][0]
    vy = global_velocity[1][0]
    omega = global_velocity[2][0]

    return vx, vy, omega

def update_position(x, y, theta, vx, vy, omega, dt=0.1):
    x_new = x + (vx * dt)
    y_new = y + (vy * dt)
    theta_new = theta + (omega * dt)

    theta_new = (theta_new + np.pi) % (2 * np.pi) - np.pi

    return x_new, y_new, theta_new

In [ ]:
def get_velocities(cureent_x, current_y, current_theta, target_x, target_y):
    dx = target_x - cureent_x
    dy = target_y - current_y
    distance = np.hypot(dx, dy)
    target_theta = np.arctan2(dy, dx)

    angle_diff = target_theta - current_theta
    angle_diff = (angle_diff + np.pi) % (2 * np.pi) - np.pi

    Kp_turn = 2.0
    Kp_drive = 1.0 

    omega = Kp_turn * angle_diff

    local_v = Kp_drive * distance
    local_v = max(0.0, local_v)

    global_v = local_v * np.cos(current_theta)
    global_vy = local_v * np.sin(current_theta)
    return global_v, global_vy, omega